# Notebook 05 — Multi-Metric Agreement Evaluation

**Purpose:** Re-evaluate the human annotation study using multiple agreement metrics, not just Cohen's κ.

**Context:** Cohen's κ is known to deflate under imbalanced label distributions — the 'paradox of high agreement, low κ.' Our gap-detection task is highly imbalanced (most (problem × KC) cells are negative), so κ under-reports the true agreement. This notebook computes three complementary metrics:

1. **Cohen's κ** — the standard metric, kept for continuity with prior work.
2. **Gwet's AC1** — chance-corrected agreement that is robust to label imbalance (Gwet, 2008).
3. **Problem-level F1** — agreement at the instructor-relevant granularity (did raters identify the same gap set for this student on this problem?).

**Inputs:** existing annotation JSON files from the human annotation study and the four LLM configurations (Exp10a Baseline, Exp10b Enriched, Exp11 Baseline V2, Exp11 Enriched V2), plus Human A and Human B.

**Outputs:** a single comparison table with all three metrics across all rater pairs, ready for inclusion in Chapter 4 of the thesis.

## Step 1 — Install dependencies

`irrCAC` is the Python port of Kilem Gwet's R package. It implements AC1, AC2, Cohen's κ, Fleiss' κ, and Krippendorff's α in a consistent interface. We use it because it is traceable to the original methodology (same author as the AC1 paper), which is defensible in thesis review.

Run this cell once; skip on subsequent runs.

## Step 2 — Imports and configuration

Define the 18 KC tags (must match the exact vocabulary used in the annotation tool and the LLM prompts). Set the paths to the annotation JSON files.

In [2]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score
from irrCAC.raw import CAC

# 18 KC tags — must match the annotation tool and prompt vocabulary exactly.
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]

# Student IDs covered in the annotation study (one per cluster)
STUDENT_IDS = ['10155', '14475', '14476']  # Struggling, High Performer, Average

# Each rater has three JSON files — one per student. They get merged at load time.
ANNOTATION_PATHS = {
    'Human_A': [
        "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_10155_1774736175604.json",
        "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14475_1775593592812.json",
        "dataset/Rater_KC_Tags/kc_annotations_Pranay Ghuge_14476_1775593175294.json",
    ],
    'Human_B': [
        "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_10155_1774820622134.json",
        "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14475_1775593132134.json",
        "dataset/Rater_KC_Tags/kc_annotations_Arundhati Das_14476_1775691915454.json",
    ],
    'Exp10a_Baseline': [
        "results/human_validation/llm_baseline_annotations_10155.json",
        "results/human_validation/llm_baseline_annotations_14475.json",
        "results/human_validation/llm_baseline_annotations_14476.json",
    ],
    'Exp10b_Enriched': [
        "results/human_validation/llm_annotations_10155.json",
        "results/human_validation/llm_annotations_14475.json",
        "results/human_validation/llm_annotations_14476.json",
    ],
    'Exp11_Baseline_V2': [
        "results/human_validation/llm_baseline_annotations_v2_10155.json",
        "results/human_validation/llm_baseline_annotations_v2_14475.json",
        "results/human_validation/llm_baseline_annotations_v2_14476.json",
    ],
    'Exp11_Enriched_V2': [
        "results/human_validation/llm_enriched_annotations_v2_10155.json",
        "results/human_validation/llm_enriched_annotations_v2_14475.json",
        "results/human_validation/llm_enriched_annotations_v2_14476.json",
    ],
}

## Step 3 — Load annotations

Each annotation file is expected to follow the schema:

```json
{
  "annotations": {
    "<problem_id>": {"gaps": ["KC1", "KC2"]},
    "<problem_id>": {"gaps": []},
    ...
  }
}
```

We load each file into a dict mapping `problem_id -> set(gap_kcs)`.

In [3]:
def load_annotations(paths: list[str]) -> dict[str, set[str]]:
    """Load and merge annotation JSON files from multiple students.
    
    Each file covers one student. We merge by namespacing problem_ids 
    with the student_id to keep them disjoint across students.
    Returns {student_id:problem_id: set(gaps)}.
    """
    merged = {}
    for path in paths:
        # Extract student_id from filename (e.g., "..._10155_..." -> "10155")
        # The student_id is the numeric segment matching one of STUDENT_IDS.
        student_id = next((sid for sid in STUDENT_IDS if f'_{sid}_' in path or f'_{sid}.' in path), None)
        if student_id is None:
            raise ValueError(f"Could not extract student_id from {path}")
        
        with open(path, 'r') as f:
            raw = json.load(f)
        anns = raw.get('annotations', raw)  # support both wrapped and flat
        
        for pid, v in anns.items():
            key = f'{student_id}:{pid}'  # namespace to avoid collision across students
            merged[key] = set(v.get('gaps', []))
    return merged

all_annotations = {name: load_annotations(paths) for name, paths in ANNOTATION_PATHS.items()}

# Common keys across all raters (student:problem pairs annotated by everyone)
common_pids = sorted(set.intersection(*[set(a.keys()) for a in all_annotations.values()]))

print(f'Loaded {len(all_annotations)} annotation sets')
for name, anns in all_annotations.items():
    by_student = {}
    for key in anns:
        sid = key.split(':')[0]
        by_student[sid] = by_student.get(sid, 0) + 1
    print(f'  {name}: {len(anns)} total — ' + ', '.join(f'{sid}:{n}' for sid, n in sorted(by_student.items())))
print(f'\nCommon (student:problem) pairs across all sets: {len(common_pids)}')

Loaded 6 annotation sets
  Human_A: 150 total — 10155:50, 14475:50, 14476:50
  Human_B: 150 total — 10155:50, 14475:50, 14476:50
  Exp10a_Baseline: 146 total — 10155:46, 14475:50, 14476:50
  Exp10b_Enriched: 146 total — 10155:46, 14475:50, 14476:50
  Exp11_Baseline_V2: 146 total — 10155:46, 14475:50, 14476:50
  Exp11_Enriched_V2: 146 total — 10155:46, 14475:50, 14476:50

Common (student:problem) pairs across all sets: 146


## Step 4 — Build the rater-by-item matrix

`irrCAC` expects a DataFrame where each row is an item (a decision the raters had to make) and each column is a rater. For our task, each 'item' is one (problem_id, KC) pair — a binary yes/no decision about whether that KC is a gap on that problem.

For N shared problems × 18 KCs, we get N × 18 rows per rater-pair comparison.

In [4]:
def build_rater_matrix(anns_x: dict, anns_y: dict, pids: list) -> pd.DataFrame:
    """Build a DataFrame with one row per (problem, KC) item and two rater columns."""
    rows = []
    for pid in pids:
        gx = anns_x.get(pid, set())
        gy = anns_y.get(pid, set())
        for kc in EXACT_KC_TAGS:
            rows.append({
                'item': f'{pid}_{kc}',
                'rater_x': 1 if kc in gx else 0,
                'rater_y': 1 if kc in gy else 0,
            })
    return pd.DataFrame(rows).set_index('item')

## Step 5 — Define the three metrics

### Cell-level Cohen's κ and Gwet's AC1

Both are computed on the flattened (problem × KC) binary vector. κ is computed via sklearn for consistency with the existing pipeline. AC1 is computed via `irrCAC`, which also returns a 95% confidence interval — useful for reporting.

### Problem-level F1

A different granularity: for each (problem, student) pair, we compare the *set* of gaps flagged by rater X against the set flagged by rater Y. For each problem we compute precision and recall over the gap sets; we then average across problems.

Special case: if both raters return an empty set for a problem, they fully agree — this counts as F1 = 1 for that problem.

In [5]:
def compute_cell_kappa(df: pd.DataFrame) -> float:
    return cohen_kappa_score(df['rater_x'], df['rater_y'])

def compute_gwet_ac1(df: pd.DataFrame) -> dict:
    """Compute Gwet's AC1 with 95% CI using irrCAC."""
    cac = CAC(df[['rater_x', 'rater_y']])
    result = cac.gwet()
    coeff = result['est']['coefficient_value']
    ci = result['est']['confidence_interval']
    return {'ac1': coeff, 'ci_low': ci[0], 'ci_high': ci[1]}

def compute_problem_level_f1(anns_x: dict, anns_y: dict, pids: list) -> dict:
    """Per-problem set F1, averaged across problems."""
    f1s, jaccards = [], []
    for pid in pids:
        gx, gy = anns_x.get(pid, set()), anns_y.get(pid, set())
        if not gx and not gy:
            f1s.append(1.0); jaccards.append(1.0); continue
        if not gx or not gy:
            f1s.append(0.0); jaccards.append(0.0); continue
        tp = len(gx & gy)
        precision = tp / len(gy) if gy else 0
        recall = tp / len(gx) if gx else 0
        f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0
        jaccard = tp / len(gx | gy)
        f1s.append(f1); jaccards.append(jaccard)
    return {
        'problem_f1_mean': np.mean(f1s),
        'jaccard_mean': np.mean(jaccards),
    }

## Step 6 — Compute metrics for all comparisons

The comparisons we need for the thesis table:

- **Baseline ceiling:** Human A vs Human B
- **LLM vs each human:** four LLM configs × two humans = 8 pairings
- **Average LLM vs Humans:** mean of the two human-LLM κ/AC1/F1 per config (matches existing pipeline)

In [6]:
def evaluate_pair(name_x: str, anns_x: dict, name_y: str, anns_y: dict, pids: list) -> dict:
    df = build_rater_matrix(anns_x, anns_y, pids)
    kappa = compute_cell_kappa(df)
    gwet = compute_gwet_ac1(df)
    plf1 = compute_problem_level_f1(anns_x, anns_y, pids)
    return {
        'Comparison': f'{name_x} vs {name_y}',
        'Cohen_kappa': kappa,
        'Gwet_AC1': gwet['ac1'],
        'AC1_CI_low': gwet['ci_low'],
        'AC1_CI_high': gwet['ci_high'],
        'Problem_F1': plf1['problem_f1_mean'],
        'Jaccard': plf1['jaccard_mean'],
    }

# Human-Human ceiling
results = [evaluate_pair('Human_A', all_annotations['Human_A'],
                          'Human_B', all_annotations['Human_B'], common_pids)]

# Each LLM config vs each human, and averaged
llm_configs = ['Exp10a_Baseline', 'Exp10b_Enriched', 'Exp11_Baseline_V2', 'Exp11_Enriched_V2']
for cfg in llm_configs:
    r_a = evaluate_pair('Human_A', all_annotations['Human_A'], cfg, all_annotations[cfg], common_pids)
    r_b = evaluate_pair('Human_B', all_annotations['Human_B'], cfg, all_annotations[cfg], common_pids)
    # Averaged row
    avg = {
        'Comparison': f'AvgHuman vs {cfg}',
        'Cohen_kappa': (r_a['Cohen_kappa'] + r_b['Cohen_kappa']) / 2,
        'Gwet_AC1': (r_a['Gwet_AC1'] + r_b['Gwet_AC1']) / 2,
        'AC1_CI_low': None, 'AC1_CI_high': None,  # CIs don't average cleanly; leave blank
        'Problem_F1': (r_a['Problem_F1'] + r_b['Problem_F1']) / 2,
        'Jaccard': (r_a['Jaccard'] + r_b['Jaccard']) / 2,
    }
    results.extend([r_a, r_b, avg])

results_df = pd.DataFrame(results)
results_df

,Comparison,Cohen_kappa,Gwet_AC1,AC1_CI_low,AC1_CI_high,Problem_F1,Jaccard
0,Human_A vs Human_B,0.574306,0.953110,0.94421,0.96201,0.869015,0.833545
1,Human_A vs Exp10a_Baseline,0.383362,0.907720,0.89483,0.92061,0.756287,0.722554
2,Human_B vs Exp10a_Baseline,0.305417,0.903900,0.89078,0.91701,0.726424,0.693708
3,AvgHuman vs Exp10a_Baseline,0.344390,0.905810,NaN,NaN,0.741356,0.708131
4,Human_A vs Exp10b_Enriched,0.417188,0.935320,0.92480,0.94585,0.792808,0.759257
5,Human_B vs Exp10b_Enriched,0.328041,0.933050,0.92238,0.94372,0.775908,0.748190
6,AvgHuman vs Exp10b_Enriched,0.372614,0.934185,NaN,NaN,0.784358,0.753724
7,Human_A vs Exp11_Baseline_V2,0.470291,0.933630,0.92290,0.94436,0.825186,0.792422
8,Human_B vs Exp11_Baseline_V2,0.382673,0.929640,0.91863,0.94066,0.796908,0.769871
9,AvgHuman vs Exp11_Baseline_V2,0.426482,0.931635,NaN,NaN,0.811047,0.781147


## Step 7 — Produce the thesis table

Filter to the rows we actually want in Chapter 4 (Human-Human ceiling + averaged LLM-vs-Human rows), round to three decimals, and format for copy-paste.

In [7]:
thesis_mask = (results_df['Comparison'] == 'Human_A vs Human_B') | \
              results_df['Comparison'].str.startswith('AvgHuman')
thesis_table = results_df[thesis_mask][
    ['Comparison', 'Cohen_kappa', 'Gwet_AC1', 'Problem_F1', 'Jaccard']
].round(3).reset_index(drop=True)

print('\n=== Thesis Table: Agreement Metrics ===\n')
print(thesis_table.to_string(index=False))
thesis_table.to_csv('results/human_llm_agreement/agreement_metrics_table.csv', index=False)
print('\nSaved to: agreement_metrics_table.csv')


=== Thesis Table: Agreement Metrics ===

                   Comparison  Cohen_kappa  Gwet_AC1  Problem_F1  Jaccard
           Human_A vs Human_B        0.574     0.953       0.869    0.834
  AvgHuman vs Exp10a_Baseline        0.344     0.906       0.741    0.708
  AvgHuman vs Exp10b_Enriched        0.373     0.934       0.784    0.754
AvgHuman vs Exp11_Baseline_V2        0.426     0.932       0.811    0.781
AvgHuman vs Exp11_Enriched_V2        0.406     0.927       0.798    0.764

Saved to: agreement_metrics_table.csv


## Step 8 — Compute ratios and write the results paragraph

For the thesis we care about how close the best LLM gets to human-human agreement on each metric. Compute the ratio (LLM / Human-Human) for each configuration so the Chapter 4 paragraph writes itself.

In [8]:
hh_row = thesis_table[thesis_table['Comparison'] == 'Human_A vs Human_B'].iloc[0]
print(f'Human-Human ceiling:')
print(f"  κ       = {hh_row['Cohen_kappa']}")
print(f"  AC1     = {hh_row['Gwet_AC1']}")
print(f"  Prob F1 = {hh_row['Problem_F1']}")
print()

llm_rows = thesis_table[thesis_table['Comparison'].str.startswith('AvgHuman')]
for _, row in llm_rows.iterrows():
    cfg = row['Comparison'].replace('AvgHuman vs ', '')
    kappa_pct = row['Cohen_kappa'] / hh_row['Cohen_kappa'] * 100 if hh_row['Cohen_kappa'] else 0
    ac1_pct = row['Gwet_AC1'] / hh_row['Gwet_AC1'] * 100 if hh_row['Gwet_AC1'] else 0
    f1_pct = row['Problem_F1'] / hh_row['Problem_F1'] * 100 if hh_row['Problem_F1'] else 0
    print(f'{cfg}:')
    print(f"  κ       = {row['Cohen_kappa']} ({kappa_pct:.1f}% of H-H)")
    print(f"  AC1     = {row['Gwet_AC1']} ({ac1_pct:.1f}% of H-H)")
    print(f"  Prob F1 = {row['Problem_F1']} ({f1_pct:.1f}% of H-H)")
    print()

Human-Human ceiling:
  κ       = 0.574
  AC1     = 0.953
  Prob F1 = 0.869

Exp10a_Baseline:
  κ       = 0.344 (59.9% of H-H)
  AC1     = 0.906 (95.1% of H-H)
  Prob F1 = 0.741 (85.3% of H-H)

Exp10b_Enriched:
  κ       = 0.373 (65.0% of H-H)
  AC1     = 0.934 (98.0% of H-H)
  Prob F1 = 0.784 (90.2% of H-H)

Exp11_Baseline_V2:
  κ       = 0.426 (74.2% of H-H)
  AC1     = 0.932 (97.8% of H-H)
  Prob F1 = 0.811 (93.3% of H-H)

Exp11_Enriched_V2:
  κ       = 0.406 (70.7% of H-H)
  AC1     = 0.927 (97.3% of H-H)
  Prob F1 = 0.798 (91.8% of H-H)



## Step 9 — Methodological note for Chapter 4

Paste the following into Chapter 4 when reporting the agreement results:

> We evaluate inter-rater agreement using three complementary metrics. Cohen's κ is reported for continuity with prior work in educational measurement. However, κ is known to deflate on tasks with imbalanced label distributions — the so-called 'paradox of high agreement, low κ' (Feinstein & Cicchetti, 1990; Gwet, 2008). This applies directly to our multi-label gap-detection task, where the majority of (problem × KC) cells are true negatives. We therefore additionally report Gwet's AC1 (Gwet, 2008), which uses a prevalence-robust chance-correction model, and problem-level F1, which measures agreement at the granularity most relevant to instructional practice (whether two raters identified the same set of gaps for a given student-problem pair). Reporting all three metrics provides transparency about the trade-offs of each and preempts the methodological concerns specific to any single metric.

### References to add to the bibliography

- Gwet, K. L. (2008). Computing inter-rater reliability and its variance in the presence of high agreement. *British Journal of Mathematical and Statistical Psychology*, 61(1), 29-48.
- Feinstein, A. R., & Cicchetti, D. V. (1990). High agreement but low kappa: I. The problems of two paradoxes. *Journal of Clinical Epidemiology*, 43(6), 543-549.

# Step 10 
## Three students, three clusters:
   10155 = Struggling (many weak skills, many gap opportunities)
   14476 = Average    (moderate weak skills)
   14475 = High Performer (near-zero gaps expected — may produce
                           degenerate κ when no gaps exist at all)

 We re-run the three-metric evaluation within each student separately
 to see how agreement varies by student ability. Instructors deploy
 gap-detection tools on struggling students first — agreement numbers
 on that subpopulation matter more than the aggregate.

In [9]:


STUDENT_CLUSTERS = {
    '10155': 'Struggling',
    '14476': 'Average',
    '14475': 'High Performer',
}

def filter_by_student(anns: dict, student_id: str) -> dict:
    """Keep only keys belonging to a given student (keys are '<sid>:<pid>')."""
    return {k: v for k, v in anns.items() if k.startswith(f'{student_id}:')}

def evaluate_pair_safe(name_x, anns_x, name_y, anns_y, pids):
    """Wrap evaluate_pair to handle degenerate cases (e.g., High Performer
    with zero gaps for both raters, which makes κ and AC1 undefined)."""
    if not pids:
        return None
    # Check whether any gaps exist at all — if not, agreement metrics are degenerate
    any_gaps = any(anns_x.get(pid) or anns_y.get(pid) for pid in pids)
    if not any_gaps:
        return {
            'Comparison': f'{name_x} vs {name_y}',
            'Cohen_kappa': None,
            'Gwet_AC1': None,
            'AC1_CI_low': None,
            'AC1_CI_high': None,
            'Problem_F1': 1.0,  # both raters agree on "no gaps" for every problem
            'Jaccard': 1.0,
            'Note': 'All empty — degenerate',
        }
    try:
        result = evaluate_pair(name_x, anns_x, name_y, anns_y, pids)
        result['Note'] = ''
        return result
    except Exception as e:
        return {
            'Comparison': f'{name_x} vs {name_y}',
            'Cohen_kappa': None, 'Gwet_AC1': None,
            'AC1_CI_low': None, 'AC1_CI_high': None,
            'Problem_F1': None, 'Jaccard': None,
            'Note': f'Error: {type(e).__name__}',
        }

# Run per-student evaluation for all rater pairs
per_cluster_rows = []
for student_id, cluster_name in STUDENT_CLUSTERS.items():
    # Filter common_pids to just this student's problems
    student_pids = [p for p in common_pids if p.startswith(f'{student_id}:')]
    n_probs = len(student_pids)

    # Human-Human
    r = evaluate_pair_safe(
        'Human_A', filter_by_student(all_annotations['Human_A'], student_id),
        'Human_B', filter_by_student(all_annotations['Human_B'], student_id),
        student_pids,
    )
    if r:
        r['Cluster'] = cluster_name
        r['Student'] = student_id
        r['N_Problems'] = n_probs
        per_cluster_rows.append(r)

    # Each LLM config: compute vs each human, then average
    for cfg in llm_configs:
        r_a = evaluate_pair_safe(
            'Human_A', filter_by_student(all_annotations['Human_A'], student_id),
            cfg, filter_by_student(all_annotations[cfg], student_id),
            student_pids,
        )
        r_b = evaluate_pair_safe(
            'Human_B', filter_by_student(all_annotations['Human_B'], student_id),
            cfg, filter_by_student(all_annotations[cfg], student_id),
            student_pids,
        )

        def safe_avg(a, b):
            if a is None or b is None:
                return None
            return (a + b) / 2

        avg = {
            'Cluster': cluster_name,
            'Student': student_id,
            'N_Problems': n_probs,
            'Comparison': f'AvgHuman vs {cfg}',
            'Cohen_kappa': safe_avg(r_a['Cohen_kappa'], r_b['Cohen_kappa']),
            'Gwet_AC1': safe_avg(r_a['Gwet_AC1'], r_b['Gwet_AC1']),
            'AC1_CI_low': None, 'AC1_CI_high': None,
            'Problem_F1': safe_avg(r_a['Problem_F1'], r_b['Problem_F1']),
            'Jaccard': safe_avg(r_a['Jaccard'], r_b['Jaccard']),
            'Note': r_a.get('Note', '') or r_b.get('Note', ''),
        }
        per_cluster_rows.append(avg)

per_cluster_df = pd.DataFrame(per_cluster_rows)

# Order columns for readability
col_order = ['Cluster', 'Student', 'N_Problems', 'Comparison',
             'Cohen_kappa', 'Gwet_AC1', 'Problem_F1', 'Jaccard', 'Note']
per_cluster_df = per_cluster_df[col_order]

# Pretty print, grouped by cluster
print('\n=== Per-Cluster Agreement Breakdown ===\n')
for cluster in ['Struggling', 'Average', 'High Performer']:
    sub = per_cluster_df[per_cluster_df['Cluster'] == cluster]
    if sub.empty:
        continue
    print(f'--- {cluster} (Student {sub.iloc[0]["Student"]}, '
          f'{sub.iloc[0]["N_Problems"]} problems) ---')
    display_cols = ['Comparison', 'Problem_F1', 'Jaccard']
    print(sub[display_cols].round(3).to_string(index=False))
    print()

per_cluster_df.to_csv('results/human_llm_agreement/agreement_metrics_by_cluster.csv', index=False)
print('Saved to: agreement_metrics_by_cluster.csv')


=== Per-Cluster Agreement Breakdown ===

--- Struggling (Student 10155, 46 problems) ---
                   Comparison  Problem_F1  Jaccard
           Human_A vs Human_B       0.890    0.851
  AvgHuman vs Exp10a_Baseline       0.657    0.613
  AvgHuman vs Exp10b_Enriched       0.785    0.741
AvgHuman vs Exp11_Baseline_V2       0.795    0.756
AvgHuman vs Exp11_Enriched_V2       0.763    0.721

--- Average (Student 14476, 50 problems) ---
                   Comparison  Problem_F1  Jaccard
           Human_A vs Human_B       0.778    0.711
  AvgHuman vs Exp10a_Baseline       0.643    0.589
  AvgHuman vs Exp10b_Enriched       0.651    0.604
AvgHuman vs Exp11_Baseline_V2       0.723    0.674
AvgHuman vs Exp11_Enriched_V2       0.711    0.656

--- High Performer (Student 14475, 50 problems) ---
                   Comparison  Problem_F1  Jaccard
           Human_A vs Human_B       0.940    0.940
  AvgHuman vs Exp10a_Baseline       0.918    0.915
  AvgHuman vs Exp10b_Enriched       0.917    0